# Phase 2 Data processing
Ce notebook prépare et nettoie les données depuis la base SQLite, calcule la matrice RFM, crée le label de churn, effectue le feature engineering et sauvegarde les jeux train/val/test dans `data/processed/`.
Objectif : pipeline reproductible pour les phases ML.

In [19]:
# Imports et configuration
import os
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path
pd.options.display.max_columns = 200
print('pandas', pd.__version__)

pandas 2.3.3


In [20]:
# Définir le chemin DB (adapter si nécessaire)
ROOT_DIR = None
cwd = Path.cwd().resolve()
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'data').exists() and (candidate / 'db').exists():
        ROOT_DIR = candidate
        break
if ROOT_DIR is None:
    raise FileNotFoundError('Impossible de trouver la racine du projet avec les dossiers data/ et db/.')
DB_PATH = ROOT_DIR / 'db' / 'retailsense.db'
if not DB_PATH.exists():
    raise FileNotFoundError(f'retailsense.db introuvable dans {DB_PATH}')
conn = sqlite3.connect(str(DB_PATH))
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print('Tables in DB:', tables['name'].tolist())


Tables in DB: ['sqlite_sequence', 'product_category_name_translation', 'geolocation', 'customers', 'sellers', 'products', 'orders', 'order_items', 'payments', 'reviews']


In [21]:
# Chargement des tables principales (si certaines tables n'existent pas, on crée des DataFrames vides)
def safe_read(table_name):
    try:
        return pd.read_sql_query(f'SELECT * FROM {table_name}', conn)
    except Exception:
        return pd.DataFrame()

customers = safe_read('customers')
orders = safe_read('orders')
order_items = safe_read('order_items')
products = safe_read('products')
reviews = safe_read('reviews')
print('shapes -> customers, orders, order_items, products, reviews:', customers.shape, orders.shape, order_items.shape, products.shape, reviews.shape)


shapes -> customers, orders, order_items, products, reviews: (99441, 5) (99441, 8) (112650, 7) (32951, 9) (99224, 7)


In [22]:
# Nettoyage basique : casts et diagnostics
def parse_date_series(s):
    return pd.to_datetime(s, errors='coerce')

if not orders.empty:
    # colonnes courantes: order_purchase_timestamp, order_approved_at
    for col in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_customer_date']:
        if col in orders.columns:
            orders[col] = parse_date_series(orders[col])
    print('\norders missing summary:')
    display(orders.isnull().sum().sort_values())

if not order_items.empty:
    order_items = order_items.drop_duplicates()
    print('\norder_items missing summary:')
    display(order_items.isnull().sum().sort_values())



orders missing summary:


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_estimated_delivery_date       0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64


order_items missing summary:


order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [23]:
# Jointure pour table analytique (one row per order_item). Adapter les clés si elles diffèrent.
df = order_items.copy()
if not orders.empty:
    df = df.merge(orders, on='order_id', how='left', suffixes=('_oi','_ord'))
if not customers.empty:
    df = df.merge(customers, on='customer_id', how='left')
if not products.empty and 'product_id' in df.columns:
    df = df.merge(products, on='product_id', how='left')
if not reviews.empty and 'order_id' in reviews.columns:
    df = df.merge(reviews, on='order_id', how='left')
print('Analytical DF shape:', df.shape)
display(df.head(3))


Analytical DF shape: (113314, 32)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,97ca439bc427b48bc1cd7177abe71365,5.0,No title,"Perfeito, produto entregue antes do combinado.",2017-09-21 00:00:00,2017-09-22 10:57:03
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,7b07bacd811c4117b742569b04ce3580,4.0,No title,No comment,2017-05-13 00:00:00,2017-05-15 11:34:13
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,0c5b33dea94867d1ac402749e5438e8b,5.0,No title,Chegou antes do prazo previsto e o produto sur...,2018-01-23 00:00:00,2018-01-23 16:06:31


In [24]:
# Calcul RFM par client
if orders.empty:
    raise ValueError('orders est vide : exécutez d abord db/init_db.py pour remplir la base')
reference_date = orders['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

# s'assurer d'avoir une colonne monétaire dans orders ; sinon sommer price depuis order_items
if 'order_total' not in orders.columns:
    if 'price' in order_items.columns:
        oi_sum = order_items.groupby('order_id').agg({'price':'sum'}).rename(columns={'price':'order_total'})
        orders = orders.merge(oi_sum, on='order_id', how='left')
        orders['order_total'] = orders['order_total'].fillna(0)
    else:
        orders['order_total'] = orders.get('order_value', 0)

rfm = orders.groupby('customer_id').agg({
    'order_purchase_timestamp': lambda x: (reference_date - pd.to_datetime(x)).min(),
    'order_id': 'nunique',
    'order_total': 'sum'
}).reset_index()
rfm.columns = ['customer_id', 'recency_timedelta', 'frequency', 'monetary']
rfm['recency'] = rfm['recency_timedelta'].dt.days
rfm = rfm.drop(columns=['recency_timedelta'])
display(rfm.head())


,customer_id,frequency,monetary,recency
0,00012a2ce6f8dcda20d059ce98491703,1,89.80,338
1,000161a058600d5901f007fab4c27140,1,54.90,459
2,0001fd6190edaaf884bcaf3d49edf079,1,179.99,597
3,0002414f95344307404f0ace7a26f1d5,1,149.90,428
4,000379cdec625522490c315e70c7a9fb,1,93.00,199


In [25]:
# Définition du label churn (paramétrique)
CHURN_DAYS = 180
rfm['churn_label'] = (rfm['recency'] > CHURN_DAYS).astype(int)
print('Churn distribution:')
display(rfm['churn_label'].value_counts())


Churn distribution:


churn_label
1    71257
0    28184
Name: count, dtype: int64

In [26]:
# Feature engineering minimal sur orders (dayofweek, month, panier total)
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'], errors='coerce')
orders['order_dow'] = orders['order_purchase_timestamp'].dt.dayofweek
orders['order_month'] = orders['order_purchase_timestamp'].dt.month
if 'order_total' not in orders.columns and 'price' in order_items.columns:
    oi_sum = order_items.groupby('order_id').agg({'price':'sum'}).rename(columns={'price':'order_total'})
    orders = orders.merge(oi_sum, on='order_id', how='left')
orders['order_total'] = orders['order_total'].fillna(0)
display(orders[['order_id','order_purchase_timestamp','order_total','order_dow']].head())


,order_id,order_purchase_timestamp,order_total,order_dow
0,00010242fe8c5a6d1ba2dd792cb16214,2017-09-13 08:59:02,58.90,2
1,00018f77f2f0320c557190d7a144bdd3,2017-04-26 10:53:06,239.90,2
2,000229ec398224ef6ca0657da4fc703e,2018-01-14 14:33:31,199.00,6
3,00024acbcdf0a6daa1e931b038114c75,2018-08-08 10:00:35,12.99,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,2017-02-04 13:57:51,199.90,5


In [27]:
# Export propre train/val/test et exemples d'exports supplémentaires (CSV)
OUT_DIR = ROOT_DIR / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def export_csv(df, name, index=False):
    path = OUT_DIR / f"{name}.csv"
    df.to_csv(path, index=index, sep=',', encoding='utf-8')
    print(f'Wrote {path} -> shape={df.shape}')

# temporal split
cut_val = orders['order_purchase_timestamp'].quantile(0.8)
cut_test = orders['order_purchase_timestamp'].quantile(0.9)
train = orders[orders['order_purchase_timestamp'] <= cut_val]
val = orders[(orders['order_purchase_timestamp'] > cut_val) & (orders['order_purchase_timestamp'] <= cut_test)]
test = orders[orders['order_purchase_timestamp'] > cut_test]

# Export : tous en CSV
export_csv(train, 'train')
export_csv(val, 'val')
export_csv(test, 'test')

# Exemples d'exports additionnels (CSV uniquement)
export_csv(rfm, 'rfm')
export_csv(orders[['order_id','customer_id','order_purchase_timestamp','order_total']], 'feature_engineering')


Wrote E:\Data\CDI_College\Cours_Profession_de_Inteligence_Artificiel\16- Projet_intérgration\RetailSenseAI\data\processed\train.csv -> shape=(79553, 11)
Wrote E:\Data\CDI_College\Cours_Profession_de_Inteligence_Artificiel\16- Projet_intérgration\RetailSenseAI\data\processed\val.csv -> shape=(9944, 11)
Wrote E:\Data\CDI_College\Cours_Profession_de_Inteligence_Artificiel\16- Projet_intérgration\RetailSenseAI\data\processed\test.csv -> shape=(9944, 11)
Wrote E:\Data\CDI_College\Cours_Profession_de_Inteligence_Artificiel\16- Projet_intérgration\RetailSenseAI\data\processed\rfm.csv -> shape=(99441, 5)
Wrote E:\Data\CDI_College\Cours_Profession_de_Inteligence_Artificiel\16- Projet_intérgration\RetailSenseAI\data\processed\feature_engineering.csv -> shape=(99441, 4)
